In [1]:
# ============================================================
# Modeling Part 1 - Simple Earnings Straddle Dataset
# ============================================================
#
# Goal:
# 1. Load the simple after-expiry straddle dataset
# 2. Create the main target variable
# 3. Add liquidity filters
# 4. Save clean simple modeling datasets
# 5. Print diagnostics to understand the data
#
# Main idea:
# We use the nearest post-earnings ATM straddle as the market's implied move.
#
# implied_move_simple = after_straddle_mid / stock_price
#
# Then compare it to the realized move:
#
# simple_straddle_edge_open = realized_move_open_to_prevclose - implied_move_simple
#
# Positive edge = stock moved more than options implied
# Negative edge = stock moved less than options implied

import os
import duckdb
from pathlib import Path

# Connect to DuckDB
con = duckdb.connect()

# Change this if your processed data folder is different
PROCESSED_DIR = r"data\processed"

# Input file created by the corrected Build Earnings Straddle Dataset notebook
simple_file = os.path.join(PROCESSED_DIR, "earnings_straddles_simple_after.parquet")

# Output files from this notebook
base_file = os.path.join(PROCESSED_DIR, "earnings_model_simple_base.parquet")
clean_loose_file = os.path.join(PROCESSED_DIR, "earnings_model_simple_clean_loose.parquet")
clean_strict_file = os.path.join(PROCESSED_DIR, "earnings_model_simple_clean_strict.parquet")

print("Input file:")
print(simple_file)

print("\nOutput files:")
print(base_file)
print(clean_loose_file)
print(clean_strict_file)

Input file:
data\processed\earnings_straddles_simple_after.parquet

Output files:
data\processed\earnings_model_simple_base.parquet
data\processed\earnings_model_simple_clean_loose.parquet
data\processed\earnings_model_simple_clean_strict.parquet


In [2]:
# Helper function to display query results neatly

def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    
    df = con.execute(query).df()
    
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

In [3]:
# Check that the simple straddle dataset exists before continuing

if not os.path.exists(simple_file):
    raise FileNotFoundError(
        f"Could not find {simple_file}. "
        "Run the corrected Build Earnings Straddle Dataset notebook first."
    )

print("File found. Ready to continue.")

File found. Ready to continue.


In [4]:
# Load the simple after-expiry straddle dataset
#
# Each row should be one earnings event with:
# - ticker
# - earnings date
# - stock price before earnings
# - ATM call and put prices
# - implied move from the straddle
# - realized stock move after earnings

con.execute(f"""
CREATE OR REPLACE TEMP TABLE simple_raw AS
SELECT *
FROM read_parquet('{simple_file}')
""")

show(
    "Raw simple straddle dataset row count",
    """
    SELECT COUNT(*) AS n_rows
    FROM simple_raw
    """
)

show(
    "Sample rows",
    """
    SELECT *
    FROM simple_raw
    LIMIT 5
    """
)


Raw simple straddle dataset row count
 n_rows
   9804

Sample rows
 gvkey  tic  permno  secid        rdq  prev_date before_exdate after_exdate  om_close  strike  strike_price  before_call_mid  before_put_mid  before_straddle_mid  after_call_mid  after_put_mid  after_straddle_mid  before_call_bid  before_call_ask  before_put_bid  before_put_ask  after_call_bid  after_call_ask  after_put_bid  after_put_ask  before_call_oi  before_put_oi  after_call_oi  after_put_oi  before_call_vol  before_put_vol  after_call_vol  after_put_vol  prev_prc  same_date  same_prc  next_date  next_prc prev_openprc same_openprc next_openprc  implied_move_simple  implied_move_isolated  realized_move_open_to_prevclose  realized_move_close_to_prevclose
015247  VLO   85269 111560 2022-01-27 2022-01-26           NaT   2022-01-28     81.30    81.0         81000              NaN             NaN                  NaN           1.630          1.340               2.970              NaN              NaN             NaN   

In [5]:
# Check whether the important columns exist and are populated.
#
# Important columns:
# implied_move_simple:
#     What the options market implied before earnings.
#
# realized_move_open_to_prevclose:
#     What the stock actually moved after earnings.
#
# after_call_mid / after_put_mid:
#     Option prices used to build the straddle.
#
# after_call_bid / after_call_ask / after_put_bid / after_put_ask:
#     Used for bid-ask spread and transaction cost checks.

show(
    "Missing values in important columns",
    """
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN implied_move_simple IS NULL THEN 1 ELSE 0 END) AS missing_implied_move_simple,
        SUM(CASE WHEN realized_move_open_to_prevclose IS NULL THEN 1 ELSE 0 END) AS missing_realized_move_open,
        SUM(CASE WHEN after_call_mid IS NULL THEN 1 ELSE 0 END) AS missing_after_call_mid,
        SUM(CASE WHEN after_put_mid IS NULL THEN 1 ELSE 0 END) AS missing_after_put_mid,
        SUM(CASE WHEN after_call_bid IS NULL THEN 1 ELSE 0 END) AS missing_after_call_bid,
        SUM(CASE WHEN after_call_ask IS NULL THEN 1 ELSE 0 END) AS missing_after_call_ask,
        SUM(CASE WHEN after_put_bid IS NULL THEN 1 ELSE 0 END) AS missing_after_put_bid,
        SUM(CASE WHEN after_put_ask IS NULL THEN 1 ELSE 0 END) AS missing_after_put_ask
    FROM simple_raw
    """
)


Missing values in important columns
 total_rows  missing_implied_move_simple  missing_realized_move_open  missing_after_call_mid  missing_after_put_mid  missing_after_call_bid  missing_after_call_ask  missing_after_put_bid  missing_after_put_ask
       9804                          0.0                         0.0                     0.0                    0.0                     0.0                     0.0                    0.0                    0.0


In [6]:
# Create the base modeling dataset.
#
# This is not the final cleaned dataset yet.
# This step creates useful variables:
#
# simple_straddle_edge_open:
#     realized move - implied move
#
# Interpretation:
#     positive = stock moved more than options implied
#     negative = stock moved less than options implied
#
# simple_straddle_edge_close:
#     same idea, but using close-to-close realized move
#
# after_call_spread_pct and after_put_spread_pct:
#     how wide the option bid-ask spread is
#
# Lower spread percentage = more liquid / cleaner option quote

con.execute("""
CREATE OR REPLACE TEMP TABLE simple_base AS
SELECT
    gvkey,
    tic,
    permno,
    secid,
    rdq,
    prev_date,
    after_exdate,

    om_close,
    strike,
    strike_price,

    after_call_mid,
    after_put_mid,
    after_straddle_mid,

    after_call_bid,
    after_call_ask,
    after_put_bid,
    after_put_ask,

    after_call_oi,
    after_put_oi,
    after_call_vol,
    after_put_vol,

    prev_prc,
    same_date,
    same_prc,
    next_date,
    next_prc,
    prev_openprc,
    same_openprc,
    next_openprc,

    implied_move_simple,
    realized_move_open_to_prevclose,
    realized_move_close_to_prevclose,

    -- Main target:
    -- Positive means the stock moved more than the straddle implied.
    realized_move_open_to_prevclose - implied_move_simple AS simple_straddle_edge_open,

    -- Secondary target:
    realized_move_close_to_prevclose - implied_move_simple AS simple_straddle_edge_close,

    -- Old sign if needed:
    -- Positive means options implied more than what realized.
    implied_move_simple - realized_move_open_to_prevclose AS simple_mispricing_open,
    implied_move_simple - realized_move_close_to_prevclose AS simple_mispricing_close,

    -- Dollar bid-ask spreads
    after_call_ask - after_call_bid AS after_call_spread,
    after_put_ask - after_put_bid AS after_put_spread,

    -- Percentage bid-ask spreads
    CASE
        WHEN after_call_mid IS NOT NULL AND after_call_mid > 0
        THEN (after_call_ask - after_call_bid) / after_call_mid
        ELSE NULL
    END AS after_call_spread_pct,

    CASE
        WHEN after_put_mid IS NOT NULL AND after_put_mid > 0
        THEN (after_put_ask - after_put_bid) / after_put_mid
        ELSE NULL
    END AS after_put_spread_pct,

    -- Time variables useful for later grouping
    EXTRACT(YEAR FROM rdq) AS earnings_year,
    EXTRACT(MONTH FROM rdq) AS earnings_month

FROM simple_raw

WHERE implied_move_simple IS NOT NULL
  AND realized_move_open_to_prevclose IS NOT NULL
  AND after_call_mid IS NOT NULL
  AND after_put_mid IS NOT NULL
  AND om_close IS NOT NULL
  AND om_close > 0
""")

show(
    "Base simple modeling dataset row count",
    """
    SELECT COUNT(*) AS n_rows
    FROM simple_base
    """
)

show(
    "Sample base rows",
    """
    SELECT
        tic,
        rdq,
        om_close,
        strike,
        after_straddle_mid,
        implied_move_simple,
        realized_move_open_to_prevclose,
        simple_straddle_edge_open,
        after_call_spread_pct,
        after_put_spread_pct,
        after_call_oi,
        after_put_oi,
        after_call_vol,
        after_put_vol
    FROM simple_base
    ORDER BY rdq, tic
    LIMIT 20
    """
)


Base simple modeling dataset row count
 n_rows
   9804

Sample base rows
 tic        rdq  om_close  strike  after_straddle_mid  implied_move_simple  realized_move_open_to_prevclose  simple_straddle_edge_open  after_call_spread_pct  after_put_spread_pct  after_call_oi  after_put_oi  after_call_vol  after_put_vol
NVDA 2018-02-08    228.80   230.0              18.875             0.082496                         0.041302                  -0.041193               0.038356              0.020513           2391          1956            3381           1754
   A 2018-02-14     68.34    67.5               2.605             0.038118                         0.096576                   0.058458               0.075362              0.181818            742           891              30             52
AMAT 2018-02-14     49.55    49.5               2.840             0.057316                         0.052674                  -0.004642               0.034602              0.035842            474           3

In [7]:
# Basic sanity checks.
#
# We want to make sure the values are reasonable.
#
# Example:
# implied_move_simple around 0.05 means 5%
# realized_move_open_to_prevclose around 0.03 means 3%
# simple_straddle_edge_open around -0.02 means realized was 2 percentage points below implied

show(
    "Basic summary of implied move, realized move, and edge",
    """
    SELECT
        COUNT(*) AS n_events,

        AVG(implied_move_simple) AS avg_implied_move,
        APPROX_QUANTILE(implied_move_simple, 0.05) AS implied_p05,
        APPROX_QUANTILE(implied_move_simple, 0.50) AS implied_median,
        APPROX_QUANTILE(implied_move_simple, 0.95) AS implied_p95,

        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        APPROX_QUANTILE(realized_move_open_to_prevclose, 0.05) AS realized_p05,
        APPROX_QUANTILE(realized_move_open_to_prevclose, 0.50) AS realized_median,
        APPROX_QUANTILE(realized_move_open_to_prevclose, 0.95) AS realized_p95,

        AVG(simple_straddle_edge_open) AS avg_edge,
        APPROX_QUANTILE(simple_straddle_edge_open, 0.05) AS edge_p05,
        APPROX_QUANTILE(simple_straddle_edge_open, 0.50) AS edge_median,
        APPROX_QUANTILE(simple_straddle_edge_open, 0.95) AS edge_p95
    FROM simple_base
    """
)


Basic summary of implied move, realized move, and edge
 n_events  avg_implied_move  implied_p05  implied_median  implied_p95  avg_realized_move  realized_p05  realized_median  realized_p95  avg_edge  edge_p05  edge_median  edge_p95
     9804          0.057391     0.029197        0.052497     0.100851           0.042261      0.002703         0.030861       0.11888  -0.01513  -0.06712    -0.019672  0.050961


In [8]:
# Check extreme values.
#
# Very large implied or realized moves may be real, but they can also indicate data problems.
# This does not delete anything yet. It only shows suspicious rows.

show(
    "Largest implied moves",
    """
    SELECT
        tic,
        rdq,
        om_close,
        strike,
        after_straddle_mid,
        implied_move_simple,
        realized_move_open_to_prevclose,
        simple_straddle_edge_open
    FROM simple_base
    ORDER BY implied_move_simple DESC
    LIMIT 20
    """
)

show(
    "Largest realized moves",
    """
    SELECT
        tic,
        rdq,
        om_close,
        prev_prc,
        next_openprc,
        implied_move_simple,
        realized_move_open_to_prevclose,
        simple_straddle_edge_open
    FROM simple_base
    ORDER BY realized_move_open_to_prevclose DESC
    LIMIT 20
    """
)


Largest implied moves
 tic        rdq  om_close  strike  after_straddle_mid  implied_move_simple  realized_move_open_to_prevclose  simple_straddle_edge_open
 KIM 2024-02-08     20.17    20.0               8.325             0.412742                         0.008924                  -0.403818
VICI 2022-05-04     29.93    30.0              11.675             0.390077                         0.029068                  -0.361009
 LHX 2019-07-31    200.00   200.0              64.950             0.324750                         0.011150                  -0.313600
 TFC 2022-01-18     67.41    67.5              21.050             0.312268                         0.004154                  -0.308115
   T 2022-04-21     19.43    19.5               6.060             0.311889                         0.031652                  -0.280237
  GE 2023-01-24     79.77    80.0              24.775             0.310580                         0.000125                  -0.310455
 DRI 2020-03-19     34.16    34.

In [9]:
# Save the base dataset.
#
# This version is lightly cleaned but does not apply strict liquidity filters yet.
# It is useful for diagnostics and future feature engineering.

if os.path.exists(base_file):
    os.remove(base_file)

con.execute(f"""
COPY simple_base
TO '{base_file}'
(FORMAT PARQUET)
""")

print(f"Saved base simple modeling dataset to:\n{base_file}")

Saved base simple modeling dataset to:
data\processed\earnings_model_simple_base.parquet


In [10]:
# Now create two cleaned versions:
#
# 1. Loose clean dataset:
#    Keeps more observations.
#    Good for early modeling and exploration.
#
# 2. Strict clean dataset:
#    Keeps only more liquid options.
#    Better for more realistic trading analysis.
#
# For the simple method, we ONLY filter on after-expiry option legs.
# We do NOT use before-expiry filters because the simple method does not use before-expiry options.

con.execute("""
CREATE OR REPLACE TEMP TABLE simple_clean_loose AS
SELECT *
FROM simple_base
WHERE
    implied_move_simple IS NOT NULL
AND realized_move_open_to_prevclose IS NOT NULL
AND simple_straddle_edge_open IS NOT NULL

-- Remove obviously unreasonable implied/realized moves.
-- These are simple safety filters.
AND implied_move_simple > 0
AND implied_move_simple < 1.00
AND realized_move_open_to_prevclose >= 0
AND realized_move_open_to_prevclose < 1.00

-- Loose bid-ask spread filters
AND after_call_spread_pct IS NOT NULL
AND after_put_spread_pct IS NOT NULL
AND after_call_spread_pct < 0.75
AND after_put_spread_pct < 0.75

-- Loose open interest filters
AND COALESCE(after_call_oi, 0) >= 1
AND COALESCE(after_put_oi, 0) >= 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE simple_clean_strict AS
SELECT *
FROM simple_base
WHERE
    implied_move_simple IS NOT NULL
AND realized_move_open_to_prevclose IS NOT NULL
AND simple_straddle_edge_open IS NOT NULL

-- Remove obviously unreasonable implied/realized moves.
AND implied_move_simple > 0
AND implied_move_simple < 1.00
AND realized_move_open_to_prevclose >= 0
AND realized_move_open_to_prevclose < 1.00

-- Stricter bid-ask spread filters
AND after_call_spread_pct IS NOT NULL
AND after_put_spread_pct IS NOT NULL
AND after_call_spread_pct < 0.50
AND after_put_spread_pct < 0.50

-- Stricter open interest filters
AND COALESCE(after_call_oi, 0) >= 10
AND COALESCE(after_put_oi, 0) >= 10

-- Volume filter
-- Same-day volume can be zero even if quote is usable,
-- but strict version requires some trading activity.
AND COALESCE(after_call_vol, 0) >= 1
AND COALESCE(after_put_vol, 0) >= 1
""")

show(
    "Row counts: base vs loose vs strict",
    """
    SELECT 'base' AS dataset, COUNT(*) AS n_rows FROM simple_base
    UNION ALL
    SELECT 'loose' AS dataset, COUNT(*) AS n_rows FROM simple_clean_loose
    UNION ALL
    SELECT 'strict' AS dataset, COUNT(*) AS n_rows FROM simple_clean_strict
    """
)


Row counts: base vs loose vs strict
dataset  n_rows
   base    9804
  loose    9022
 strict    7412


In [11]:
# Compare loose and strict datasets.
#
# We want to know:
# - how many rows survive
# - whether the average implied move changes a lot
# - whether the average realized move changes a lot
# - whether the average edge changes a lot

show(
    "Loose vs strict summary",
    """
    SELECT
        'loose' AS dataset,
        COUNT(*) AS n_events,
        COUNT(DISTINCT tic) AS n_tickers,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge,
        APPROX_QUANTILE(simple_straddle_edge_open, 0.50) AS median_edge
    FROM simple_clean_loose

    UNION ALL

    SELECT
        'strict' AS dataset,
        COUNT(*) AS n_events,
        COUNT(DISTINCT tic) AS n_tickers,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge,
        APPROX_QUANTILE(simple_straddle_edge_open, 0.50) AS median_edge
    FROM simple_clean_strict
    """
)


Loose vs strict summary
dataset  n_events  n_tickers  avg_implied_move  avg_realized_move  avg_edge  median_edge
  loose      9022        474          0.057131           0.042723 -0.014408    -0.019029
 strict      7412        457          0.057282           0.043692 -0.013590    -0.017661


In [12]:
# Check how many earnings events we have by year.
#
# This matters because later we will do time-based out-of-sample testing.
# For example:
# train on 2018-2020, test on 2021, etc.

show(
    "Year coverage: loose dataset",
    """
    SELECT
        earnings_year,
        COUNT(*) AS n_events,
        COUNT(DISTINCT tic) AS n_tickers,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge
    FROM simple_clean_loose
    GROUP BY earnings_year
    ORDER BY earnings_year
    """
)

show(
    "Year coverage: strict dataset",
    """
    SELECT
        earnings_year,
        COUNT(*) AS n_events,
        COUNT(DISTINCT tic) AS n_tickers,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge
    FROM simple_clean_strict
    GROUP BY earnings_year
    ORDER BY earnings_year
    """
)


Year coverage: loose dataset
 earnings_year  n_events  n_tickers  avg_implied_move  avg_realized_move  avg_edge
          2018       967        362          0.052866           0.041631 -0.011236
          2019      1322        417          0.052170           0.038406 -0.013763
          2020      1217        411          0.062592           0.041301 -0.021291
          2021      1253        397          0.051277           0.035110 -0.016166
          2022      1308        420          0.064164           0.045565 -0.018599
          2023      1429        434          0.055452           0.044910 -0.010541
          2024      1526        454          0.060130           0.050056 -0.010074

Year coverage: strict dataset
 earnings_year  n_events  n_tickers  avg_implied_move  avg_realized_move  avg_edge
          2018       807        330          0.052758           0.042146 -0.010612
          2019      1108        383          0.052941           0.039906 -0.013034
          2020       915  

In [13]:
# Check whether a few tickers dominate the dataset.
#
# If one ticker appears too many times, the model may mostly learn that ticker.
# This does not mean we remove anything now.
# It is just a diagnostic.

show(
    "Most common tickers in loose dataset",
    """
    SELECT
        tic,
        COUNT(*) AS n_events,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge
    FROM simple_clean_loose
    GROUP BY tic
    ORDER BY n_events DESC
    LIMIT 30
    """
)

show(
    "Most common tickers in strict dataset",
    """
    SELECT
        tic,
        COUNT(*) AS n_events,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge
    FROM simple_clean_strict
    GROUP BY tic
    ORDER BY n_events DESC
    LIMIT 30
    """
)


Most common tickers in loose dataset
 tic  n_events  avg_implied_move  avg_realized_move  avg_edge
NVDA        28          0.076669           0.050658 -0.026011
 CRM        28          0.067534           0.065208 -0.002326
  MU        28          0.076917           0.058882 -0.018035
LULU        28          0.086146           0.066823 -0.019323
 BBY        28          0.078522           0.064636 -0.013885
CSCO        28          0.051407           0.050311 -0.001096
 CCL        28          0.082523           0.065575 -0.016947
ULTA        28          0.075126           0.070891 -0.004235
 WMT        28          0.043461           0.038274 -0.005187
 LOW        28          0.051141           0.044485 -0.006656
ADBE        28          0.057129           0.053376 -0.003753
  DG        28          0.061404           0.065893  0.004489
 TGT        28          0.066505           0.088288  0.021782
  DE        28          0.052007           0.046019 -0.005988
ADSK        28          0.069062

In [14]:
# Save the clean datasets.
#
# Use loose dataset for early research and modeling.
# Use strict dataset for robustness and more realistic liquidity assumptions.

for f in [clean_loose_file, clean_strict_file]:
    if os.path.exists(f):
        os.remove(f)

con.execute(f"""
COPY simple_clean_loose
TO '{clean_loose_file}'
(FORMAT PARQUET)
""")

con.execute(f"""
COPY simple_clean_strict
TO '{clean_strict_file}'
(FORMAT PARQUET)
""")

print("Saved clean datasets:")
print(clean_loose_file)
print(clean_strict_file)

Saved clean datasets:
data\processed\earnings_model_simple_clean_loose.parquet
data\processed\earnings_model_simple_clean_strict.parquet


In [15]:
# Final summary.
#
# These are the main numbers you should look at after running this notebook.

show(
    "Final dataset summary",
    """
    SELECT
        'base' AS dataset,
        COUNT(*) AS n_events,
        COUNT(DISTINCT tic) AS n_tickers,
        MIN(rdq) AS start_date,
        MAX(rdq) AS end_date,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge
    FROM simple_base

    UNION ALL

    SELECT
        'loose' AS dataset,
        COUNT(*) AS n_events,
        COUNT(DISTINCT tic) AS n_tickers,
        MIN(rdq) AS start_date,
        MAX(rdq) AS end_date,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge
    FROM simple_clean_loose

    UNION ALL

    SELECT
        'strict' AS dataset,
        COUNT(*) AS n_events,
        COUNT(DISTINCT tic) AS n_tickers,
        MIN(rdq) AS start_date,
        MAX(rdq) AS end_date,
        AVG(implied_move_simple) AS avg_implied_move,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move,
        AVG(simple_straddle_edge_open) AS avg_edge
    FROM simple_clean_strict
    """
)


Final dataset summary
dataset  n_events  n_tickers start_date   end_date  avg_implied_move  avg_realized_move  avg_edge
   base      9804        480 2018-02-08 2024-12-20          0.057391           0.042261 -0.015130
  loose      9022        474 2018-02-08 2024-12-20          0.057131           0.042723 -0.014408
 strict      7412        457 2018-02-08 2024-12-20          0.057282           0.043692 -0.013590
